# DACIL-WESENSE Quantitative Analysis Pipeline

This notebook implements an automated, reproducible pipeline for processing,
analysing, and modelling physiological stress-test data.

**Pipeline overview:**

1. Environment setup and batch folder discovery
2. Data loading: CSV telemetry and BDF ECG files
3. Data synchronisation and cleaning
4. Exploratory Data Analysis (EDA)
5. Unsupervised Machine Learning (PCA, K-Means, DBSCAN)
6. Per-patient output export
7. Master summary CSV and notebook export


## 1. Environment Setup


In [2]:
if True:
    !pip install -r requirements.txt

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 14.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 22.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 13.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 25.7 MB/s  0:00:00 eta 0:00:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 22.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 9

In [3]:
import logging
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import functions as fn

warnings.filterwarnings("ignore")

# Configure logging: messages appear both in the notebook and in a log file.
fn.setup_logging(log_file="pipeline.log", level=logging.INFO)
logger = logging.getLogger("main")

logger.info("Pipeline started.")


2026-03-04T16:11:11 | INFO     | main | Pipeline started.


## 2. Configuration

Set `DATA_ROOT` to the directory that contains one sub-folder per patient/trial.
All outputs are written under `OUTPUT_ROOT`.


In [4]:
# ----------------------------------------------------------------
# CONFIGURE THESE PATHS BEFORE RUNNING
# ----------------------------------------------------------------
DATA_ROOT: str = "data"          # root directory with patient sub-folders
OUTPUT_ROOT: str = "output"      # base output directory
# ----------------------------------------------------------------

output_base = Path(OUTPUT_ROOT)
output_base.mkdir(parents=True, exist_ok=True)

logger.info("Data root: %s", DATA_ROOT)
logger.info("Output root: %s", OUTPUT_ROOT)


2026-03-04T16:11:11 | INFO     | main | Data root: data
2026-03-04T16:11:11 | INFO     | main | Output root: output


## 3. Batch Folder Discovery

All immediate sub-directories of `DATA_ROOT` are treated as independent
patient/trial folders.  A `try-except` block around each folder ensures
that a single problematic folder does not abort the entire batch.


In [19]:
patient_folders = fn.discover_patient_folders(DATA_ROOT)
print(f"Found {len(patient_folders)} patient folder(s).")
for pf in patient_folders:
    print(" -", pf.name)


2026-03-04T16:13:02 | INFO     | functions | Discovered 1 patient folder(s) under 'data'.


Found 1 patient folder(s).
 - Patient 1


## 4. Batch Processing Loop

For each patient folder the pipeline:

1. Loads CSV telemetry and patient metadata.
2. Loads BDF ECG files (L1 and L2) and extracts summary ECG features.
3. Synchronises ECG features with the telemetry time series.
4. Saves the cleaned telemetry as a CSV.
5. Appends a summary row to the master list.

Any folder that raises an exception is logged and skipped gracefully.


In [20]:
master_records = []
processed_telemetry: dict = {}  # patient_id -> cleaned telemetry DataFrame

for folder in patient_folders:
    patient_id = folder.name
    logger.info("Processing patient: %s", patient_id)

    try:
        # ── 4a. Load CSV ──────────────────────────────────────────────────
        csv_path = fn.find_csv_file(folder)
        if csv_path is None:
            raise FileNotFoundError(f"No CSV file found in {folder}")

        info_df, telemetry_df = fn.load_telemetry(csv_path)
        logger.info("  Telemetry shape: %s", telemetry_df.shape)

        # ── 4b. Load BDF ECG files ─────────────────────────────────────────
        l1_path, l2_path = fn.find_bdf_files(folder)
        ecg_features_list = []

        for bdf_label, bdf_path in [("L1", l1_path), ("L2", l2_path)]:
            if bdf_path is None:
                logger.warning("  %s BDF not found for %s.", bdf_label, patient_id)
                continue
            raw = fn.load_ecg(bdf_path)
            if raw is not None:
                feats = fn.extract_ecg_features(raw)
                feats["bdf_label"] = bdf_label
                ecg_features_list.append(feats)

        ecg_features = (
            pd.concat(ecg_features_list, ignore_index=True)
            if ecg_features_list
            else pd.DataFrame()
        )

        # ── 4c. Synchronise ECG + telemetry ───────────────────────────────
        telemetry_df = fn.sync_ecg_with_telemetry(ecg_features, telemetry_df)

        # ── 4d. Save cleaned CSV ───────────────────────────────────────────
        patient_out = output_base / patient_id
        fn.save_telemetry_csv(telemetry_df, patient_id, patient_out)

        # ── 4e. Build summary row ──────────────────────────────────────────
        summary_row = fn.build_patient_summary(
            patient_id, info_df, telemetry_df, ecg_features if not ecg_features.empty else None
        )
        master_records.append(summary_row)
        processed_telemetry[patient_id] = telemetry_df

        logger.info("  Successfully processed patient: %s", patient_id)

    except Exception as exc:
        logger.error("  FAILED for patient '%s': %s", patient_id, exc, exc_info=True)

print(f"Batch complete. Successfully processed {len(master_records)} patient(s).")


2026-03-04T16:13:02 | INFO     | main | Processing patient: Patient 1
2026-03-04T16:13:02 | ERROR    | main |   FAILED for patient 'Patient 1': Unsupported format, or corrupt file: Expected BOF record; found b'Identifi'
Traceback (most recent call last):
  File "/tmp/ipykernel_62936/1661387965.py", line 14, in <module>
    info_df, telemetry_df = fn.load_telemetry(xls_path)
                            ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/home/peipr/gits/quantitative_analysis/functions.py", line 149, in load_telemetry
    xls = pd.ExcelFile(xls_path, engine=_excel_engine(xls_path))
  File "/home/peipr/miniconda3/envs/quantitative_analysis/lib/python3.13/site-packages/pandas/io/excel/_base.py", line 1567, in __init__
    self._reader = self._engines[engine](
                   ~~~~~~~~~~~~~~~~~~~~~^
        self._io,
        ^^^^^^^^^
        storage_options=storage_options,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        engine_kwargs=engine_kwargs,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Batch complete. Successfully processed 0 patient(s).


## 5. Master Summary DataFrame

Compile all per-patient summary rows into a single master DataFrame and
save it as `output/master_summary.csv`.


In [21]:
if master_records:
    summary_df = pd.DataFrame(master_records)
    master_csv = output_base / "master_summary.csv"
    summary_df.to_csv(master_csv, index=False)
    logger.info("Saved master summary: %s", master_csv)
    display(summary_df.head())
else:
    summary_df = pd.DataFrame()
    print("No patient data available. Check that DATA_ROOT contains valid folders.")


No patient data available. Check that DATA_ROOT contains valid folders.


## 6. Exploratory Data Analysis

### 6a. Per-patient stage profiles

For each successfully processed patient we plot how the key physiological
metrics (VO2, HR, Power, VCO2) evolve across the five trial stages.
Figures are saved under `output/<patient_id>/`.


In [22]:
EDA_METRICS = ["HR", "VO2", "VCO2", "Power", "RER", "SpO2"]

for patient_id, telemetry_df in processed_telemetry.items():
    patient_out = output_base / patient_id
    fig = fn.plot_metrics_by_stage(
        telemetry_df,
        metrics=EDA_METRICS,
        patient_id=patient_id,
        output_dir=patient_out,
    )
    plt.show()
    plt.close(fig)


### 6b. Stage-level summary statistics

Mean and standard deviation of key metrics for each trial stage.


In [23]:
for patient_id, telemetry_df in processed_telemetry.items():
    print(f"\n--- Patient: {patient_id} ---")
    stage_summary = fn.compute_stage_summary(telemetry_df, metrics=EDA_METRICS)
    display(stage_summary)


### 6c. Batch aggregate visualisations

Aggregated views across all patients:

- Distribution of peak VO2 grouped by gender.
- BMI vs peak VO2 scatter plot.


In [24]:
if not summary_df.empty:
    batch_figs = fn.plot_batch_aggregates(summary_df, output_dir=output_base)
    for fig in batch_figs:
        plt.show()
        plt.close(fig)
else:
    print("No summary data available for batch EDA.")


No summary data available for batch EDA.


## 7. Unsupervised Machine Learning

### 7a. Rationale

**PCA** is applied to reduce the high-dimensional telemetry feature space to
a small number of interpretable axes.  The Scree plot guides the choice of
how many components capture the majority of variance (target: >= 90 %).

**K-Means** is then applied to the original scaled space.  K-Means is chosen
because the expected physiological response groups (e.g., low/moderate/high
exertion) are approximately convex and of similar size - well-suited to
centroid-based clustering.  The Elbow plot is used to select K.

**DBSCAN** is applied as a complementary density-based algorithm, which can
identify irregularly shaped clusters and flag outlier observations as noise
(label = -1).  This helps reveal atypical physiological responses that
K-Means would force into a cluster.

Results are visualised in the PCA reduced space and as VO2 vs HR scatter
plots coloured by cluster assignment.


### 7b. Feature preparation


In [25]:
# Concatenate all patient telemetry for a global ML analysis.
if processed_telemetry:
    all_tele = pd.concat(
        [df.assign(patient_id=pid) for pid, df in processed_telemetry.items()],
        ignore_index=True,
    )

    X_scaled, feature_df, scaler = fn.prepare_features(
        all_tele, drop_cols=["patient_id"]
    )
    stage_labels = all_tele["Stage"].fillna("Unknown").reset_index(drop=True)

    print(f"Feature matrix shape: {X_scaled.shape}")
else:
    print("No telemetry data available. Skipping ML section.")
    X_scaled = None


No telemetry data available. Skipping ML section.


### 7c. PCA


In [26]:
if X_scaled is not None and X_scaled.shape[0] > 1:
    pca_model, X_pca = fn.run_pca(X_scaled, n_components=min(10, X_scaled.shape[1]))

    # Scree plot
    fig_scree = fn.plot_scree(pca_model, output_dir=output_base)
    plt.show()
    plt.close(fig_scree)

    # 2-D PCA scatter coloured by trial stage
    fig_pca_2d = fn.plot_pca_scatter(
        X_pca, stage_labels, label_name="Stage", output_dir=output_base
    )
    plt.show()
    plt.close(fig_pca_2d)

    # 3-D PCA scatter coloured by trial stage (when >= 3 PCs available)
    if X_pca.shape[1] >= 3:
        fig_pca_3d = fn.plot_pca_scatter(
            X_pca,
            stage_labels,
            label_name="Stage",
            output_dir=output_base,
            three_d=True,
        )
        plt.show()
        plt.close(fig_pca_3d)
else:
    print("Insufficient data for PCA.")


Insufficient data for PCA.


### 7d. Elbow plot for K-Means


In [27]:
if X_scaled is not None and X_scaled.shape[0] > 10:
    max_k = min(10, X_scaled.shape[0] - 1)
    fig_elbow = fn.elbow_plot(
        X_scaled, k_range=range(2, max_k + 1), output_dir=output_base
    )
    plt.show()
    plt.close(fig_elbow)
else:
    print("Insufficient data for elbow analysis.")


Insufficient data for elbow analysis.


### 7e. K-Means clustering

Based on the elbow plot above, select `N_CLUSTERS`.


In [28]:
N_CLUSTERS = 4  # adjust based on the elbow plot

if X_scaled is not None and X_scaled.shape[0] >= N_CLUSTERS:
    km_model, km_labels = fn.run_kmeans(X_scaled, n_clusters=N_CLUSTERS)

    # Determine the best available metric pair for visualisation.
    vo2_col = fn._find_column(feature_df, ["VO2", "vo2", "VO2_max"])
    hr_col = fn._find_column(feature_df, ["HR", "hr", "Heart Rate"])
    x_vis = vo2_col or feature_df.columns[0]
    y_vis = hr_col or feature_df.columns[min(1, len(feature_df.columns) - 1)]

    fig_km = fn.plot_cluster_scatter(
        feature_df,
        km_labels,
        x_col=x_vis,
        y_col=y_vis,
        algorithm_name="K-Means",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_km)

    # PCA scatter coloured by K-Means label
    km_series = pd.Series(km_labels.astype(str), name="KMeans_Cluster")
    fig_pca_km = fn.plot_pca_scatter(
        X_pca,
        km_series,
        label_name="K-Means Cluster",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_pca_km)
else:
    print("Insufficient data for K-Means.")


Insufficient data for K-Means.


**Cluster interpretation:** Each cluster corresponds to a distinct
physiological response regime identifiable in the VO2 vs HR space.
Clusters with high VO2 and high HR typically represent maximal or near-maximal
exertion (Test / VT2 stages), while low-VO2 / low-HR clusters correspond to
warm-up (Opwarmen) and recovery (Herstel).


### 7f. DBSCAN clustering

DBSCAN does not require specifying K in advance and naturally identifies noise
points (label = -1), making it suitable for detecting physiologically
atypical observations.


In [29]:
if X_scaled is not None and X_scaled.shape[0] >= 10:
    db_model, db_labels = fn.run_dbscan(X_scaled, eps=1.0, min_samples=5)
    n_clusters_found = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = int(np.sum(db_labels == -1))
    print(f"DBSCAN found {n_clusters_found} cluster(s) and {n_noise} noise point(s).")

    fig_db = fn.plot_cluster_scatter(
        feature_df,
        db_labels,
        x_col=x_vis,
        y_col=y_vis,
        algorithm_name="DBSCAN",
        output_dir=output_base,
    )
    plt.show()
    plt.close(fig_db)
else:
    print("Insufficient data for DBSCAN.")


Insufficient data for DBSCAN.


**DBSCAN interpretation:** Points labelled -1 (noise) are physiological
observations that do not fit any dense neighbourhood and may indicate artefacts
or genuinely extreme responses.  The remaining clusters can be compared to
the K-Means assignment to assess stability of the groupings.


## 8. Output Summary

The `output/` directory now contains:

```
output/
  master_summary.csv
  batch_vo2_by_gender.png
  batch_bmi_vs_vo2.png
  scree.png
  pca_2d_stage.png
  elbow.png
  kmeans_clusters.png
  dbscan_clusters.png
  <patient_id>/
    <patient_id>_telemetry.csv
    <patient_id>_metrics_by_stage.png
```


In [30]:
# List all files written to the output directory.
for path in sorted(output_base.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_base))


main_report.html


## 9. Notebook Export

Export this notebook to HTML or PDF using `nbconvert`:

**HTML (recommended - no additional LaTeX required):**

```bash
jupyter nbconvert --to html main.ipynb --output output/main_report.html
```

**PDF (requires LaTeX / pandoc to be installed):**

```bash
jupyter nbconvert --to pdf main.ipynb --output output/main_report.pdf
```

Or run programmatically from within the notebook:


In [31]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "html",
        "main.ipynb",
        "--output", str(output_base / "main_report.html"),
    ],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("Notebook exported to:", output_base / "main_report.html")
else:
    print("Export failed:", result.stderr)


Notebook exported to: output/main_report.html


---

**End of pipeline.**

All outputs adhere to FAIR principles:

- **Findable:** Structured `output/` directory with consistent naming.
- **Accessible:** Standard CSV and PNG formats; HTML report for easy sharing.
- **Interoperable:** Pandas CSV format readable by any tabular data tool.
- **Reusable:** Modular `functions.py` with full docstrings and type hints;
  `requirements.txt` with pinned dependency versions.
